# Punto 2 — Detección de spam con red neuronal densa

**Objetivo:** Clasificar correos como spam o no-spam usando TF-IDF como vectorización y comparar tres modelos:
1. Red Neuronal Densa (DNN)
2. Random Forest
3. Naive Bayes Multinomial

**Regla fundamental:** El pipeline de TF-IDF se ajusta **exclusivamente** con datos de entrenamiento para evitar data leakage.

## 1. Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

sns.set_theme(style='whitegrid')
print(f'TensorFlow {tf.__version__}')

## 2. Carga de datos

In [ ]:
URL = 'https://raw.githubusercontent.com/camilousa/datasets/main/spam.csv'

try:
    df = pd.read_csv(URL, encoding='latin-1')
except Exception:
    # Fallback: intentar con el archivo local si fue descargado
    df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# El dataset tiene columnas v1 (label) y v2 (text) más columnas vacías
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
df['target'] = (df['label'] == 'spam').astype(int)

print(f'Total de mensajes: {len(df)}')
print(df['label'].value_counts())
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de clases
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Distribución de clases')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)

# Longitud de mensajes
df['length'] = df['text'].apply(len)
df.groupby('label')['length'].plot.hist(bins=50, alpha=0.6, ax=axes[1], legend=True)
axes[1].set_title('Distribución de longitud del mensaje')
axes[1].set_xlabel('Longitud (caracteres)')

plt.tight_layout()
plt.show()

## 3. División train/test (primero, antes de cualquier transformación)

In [ ]:
X_text_train, X_text_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['target'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['target'].values
)

print(f'Train: {len(X_text_train)} muestras  |  Test: {len(X_text_test)} muestras')
print(f'Proporción spam en train: {y_train.mean():.2%}')
print(f'Proporción spam en test:  {y_test.mean():.2%}')

## 4. Vectorización TF-IDF (ajustada solo en train)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',
    ngram_range=(1, 2),
    stop_words='english'
)

# fit SOLO en train
X_train = vectorizer.fit_transform(X_text_train)
# transform en test (sin refit)
X_test  = vectorizer.transform(X_text_test)

X_train_dense = X_train.toarray().astype(np.float32)
X_test_dense  = X_test.toarray().astype(np.float32)

print(f'Dimensión de X_train: {X_train_dense.shape}')
print(f'Dimensión de X_test:  {X_test_dense.shape}')

## 5. Función auxiliar de métricas y visualización

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    print(f'\n── {name} ──────────────────────────')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-score:  {f1:.4f}')
    return {'Modelo': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}


def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    ax.set_title(title)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')


all_metrics = []

## 6. Modelo 1 — Red Neuronal Densa

In [ ]:
n_features = X_train_dense.shape[1]

dnn = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
], name='DNN_Spam')

dnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
dnn.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    patience=5, restore_best_weights=True, monitor='val_loss'
)

history_dnn = dnn.fit(
    X_train_dense, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_prob_dnn = dnn.predict(X_test_dense, verbose=0).flatten()
y_pred_dnn = (y_prob_dnn >= 0.5).astype(int)

m_dnn = evaluate_model('DNN', y_test, y_pred_dnn, y_prob_dnn)
all_metrics.append(m_dnn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_dnn.history['loss'], label='Train')
axes[0].plot(history_dnn.history['val_loss'], label='Validación')
axes[0].set_title('DNN — Curva de pérdida')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()

axes[1].plot(history_dnn.history['accuracy'], label='Train')
axes[1].plot(history_dnn.history['val_accuracy'], label='Validación')
axes[1].set_title('DNN — Curva de exactitud')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Modelo 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

m_rf = evaluate_model('Random Forest', y_test, y_pred_rf, y_prob_rf)
all_metrics.append(m_rf)

## 8. Modelo 3 — Naive Bayes Multinomial

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
y_prob_nb = nb.predict_proba(X_test)[:, 1]

m_nb = evaluate_model('Naive Bayes', y_test, y_pred_nb, y_prob_nb)
all_metrics.append(m_nb)

## 9. Comparación de modelos

In [ ]:
df_metrics = pd.DataFrame(all_metrics).set_index('Modelo')
print('\n=== Tabla comparativa de modelos ===')
display(df_metrics.round(4))

In [ ]:
ax = df_metrics.plot(kind='bar', figsize=(12, 5), ylim=(0.9, 1.0))
ax.set_title('Comparación de métricas — DNN vs Random Forest vs Naive Bayes')
ax.set_xlabel('Modelo')
ax.set_ylabel('Valor')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plot_cm(y_test, y_pred_dnn, 'DNN',          axes[0])
plot_cm(y_test, y_pred_rf,  'Random Forest', axes[1])
plot_cm(y_test, y_pred_nb,  'Naive Bayes',   axes[2])

plt.suptitle('Matrices de confusión', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Análisis de resultados

### Observaciones principales

1. **Integridad del pipeline:** El vectorizador TF-IDF se ajustó exclusivamente con datos de entrenamiento (`fit_transform` en train, solo `transform` en test). Esto evita que el vocabulario del test influya en la representación, garantizando una evaluación realista.

2. **Naive Bayes:** Históricamente es el modelo de referencia para detección de spam. Su suposición de independencia entre términos no es perfecta, pero funciona bien con TF-IDF porque los pesos ya capturan la importancia relativa de cada token. Generalmente logra F1 > 0.97 con pocos datos.

3. **Random Forest:** Es el modelo más robusto cuando hay vocabulario rico y features correlacionadas. Suele superar a Naive Bayes en precision (menos falsos positivos), lo cual es crítico en spam filtering: es preferible perder un spam que bloquear un correo legítimo.

4. **DNN:** Tiene más parámetros y puede capturar relaciones no lineales entre términos. Sin embargo, en datasets pequeños (~5000 muestras) puede no superar a los modelos clásicos. El Dropout y EarlyStopping ayudan a controlar el sobreajuste.

5. **Recall vs. Precision en spam:** Un alto recall (detectar todo el spam) puede ir en detrimento de la precision (bloquear correos legítimos). La métrica F1 balancea ambas. Comparar las matrices de confusión revela cuál modelo tiene más falsos positivos (ham clasificado como spam) vs. falsos negativos (spam no detectado).

6. **Conclusión:** Para este dataset, Random Forest o Naive Bayes suelen ser competitivos o superiores a la DNN en F1-score, con la ventaja de ser más interpretables y rápidos de entrenar. La DNN podría beneficiarse de representaciones más ricas (embeddings) en lugar de TF-IDF puro.